# Optimizers: Momentum, RMSprop, Adam

_Deep Learning_

**Smarter ways to step downhill so training is faster and steadier.**

Plain gradient descent takes one fixed step downhill at a time. It can be slow and shaky.

     Optimizers are upgrades that make the steps smarter.

---

This notebook is a practice scaffold for the **Optimizers: Momentum, RMSprop, Adam** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. These are **images, not table columns** — each sample is an 8x8 grid of pixel intensities (0–16).

In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
print("image array:", digits.images.shape, " labels:", digits.target.shape)
samples = list(zip(digits.images, digits.target))
fig, axes = plt.subplots(1, 5, figsize=(8, 2))
for ax, (image, label) in zip(axes, samples):
    ax.imshow(image, cmap="gray")
    ax.set_title(str(label))
    ax.axis("off")
plt.show()

## Reference implementation — PyTorch

We wire up the smallest possible training loop and let one optimizer drive it. The point is to see that Momentum, RMSprop, and Adam are drop-in swaps: they all share the same `.step()` interface, so only one line changes between them.

### Step 1 — Build a tiny model and pick an optimizer

We need three things before we can train: a **model** (a single linear layer), a **loss function** (mean-squared error), and an **optimizer**. The optimizer is the part that decides *how* to turn gradients into weight updates. Momentum keeps a running velocity so it rolls through shallow spots; RMSprop scales each weight's step by its recent gradient size; Adam combines both ideas. Uncomment exactly one — they are interchangeable.

In [ ]:
import torch
import torch.nn as nn

model = nn.Linear(10, 1)
loss_fn = nn.MSELoss()

# pick ONE optimizer (all share the same .step() interface):
opt = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9)   # Momentum
# opt = torch.optim.RMSprop(model.parameters(), lr=0.01)          # RMSprop
# opt = torch.optim.Adam(model.parameters(), lr=0.001)            # Adam (default)

### Step 2 — Make a synthetic batch

To exercise the optimizer we just need some inputs and targets. Here `X` is 64 examples of 10 random features each, and `y` is 64 random target values. There is no real pattern to learn — we only want to watch the loss respond as the optimizer updates the weights.

In [ ]:
X = torch.randn(64, 10)   # synthetic batch: 64 examples, 10 features
y = torch.randn(64, 1)    # synthetic targets, one per example

### Step 3 — Run the training loop

Each step repeats the same four moves: zero out old gradients, compute the loss, back-propagate to get fresh gradients, then let the optimizer update the weights. That last `opt.step()` is where the chosen optimizer's behaviour kicks in. Watch the printed loss shrink over the five steps.

In [ ]:
for step in range(5):                 # short training loop
    opt.zero_grad()                   # clear gradients from the previous step
    predictions = model(X)            # forward pass
    loss = loss_fn(predictions, y)    # how wrong are we?
    loss.backward()                   # gradients
    opt.step()                        # optimizer updates the weights
    print("step", step, "loss", round(loss.item(), 4))

## Visualize the data & results

_Training a real digit classifier, which optimizer drives the loss down fastest?_

### Step 1 — Load the digit data

We move from synthetic noise to a real classification task: the 8x8 handwritten digits. Each image is flattened into 64 features and divided by 16 so every pixel sits between 0 and 1 — that rescaling keeps the optimizers' step sizes sensible.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.neural_network import MLPClassifier

digits = load_digits()
X = digits.data / 16.0   # flatten pixels to [0, 1]
y = digits.target

### Step 2 — Train one network per optimizer

To compare optimizers fairly we train the *same* small network (one hidden layer of 32 units) three times, changing only the solver and its momentum. `loss_curve_` records the training loss after every epoch, so the returned list is exactly the learning curve we want to plot.

In [ ]:
def loss_curve(solver, lr, mom):
    model = MLPClassifier(
        hidden_layer_sizes=(32,),
        solver=solver,
        max_iter=40,
        random_state=0,
        alpha=1e-4,
        learning_rate_init=lr,
        momentum=mom,
        nesterovs_momentum=False,
        batch_size=64,
    )
    model.fit(X, y)
    return model.loss_curve_

sgd_curve = loss_curve("sgd", 0.05, 0.0)        # plain SGD
momentum_curve = loss_curve("sgd", 0.05, 0.9)   # SGD + Momentum
adam_curve = loss_curve("adam", 0.01, 0.9)      # Adam

### Step 3 — Plot the three learning curves

Overlaying the curves shows the payoff: plain SGD descends slowly, Momentum speeds it up, and Adam usually reaches a low loss fastest of all. The y-axis is log loss, so lower is better.

In [ ]:
plt.plot(sgd_curve, color="#4ea1ff", label="SGD")
plt.plot(momentum_curve, color="#ffb454", label="SGD+Momentum")
plt.plot(adam_curve, color="#7ee787", label="Adam")
plt.title("Real training loss by optimizer on load_digits")
plt.xlabel("epoch")
plt.ylabel("log loss")
plt.legend()
plt.show()